This is part of the pipeline I used to generate the following dataset with 3k+ essays

https://www.kaggle.com/datasets/illidan7/pii-detect-illi-train-dataset

_____

- Explore the competition dataset to better understand the structure of the essays and the PIIs in the data

https://www.kaggle.com/code/illidan7/pii-detect-data-explore

- Generate PIIs; Generate PII targets that closely resembled the ones in the competition data

https://www.kaggle.com/code/illidan7/pii-detect-mistral-pii-generation/notebook
https://www.kaggle.com/datasets/illidan7/pii-detect-generated-piis

- Generate Essays; Generate essays similar in structure to the competition data

https://www.kaggle.com/code/illidan7/pii-detect-mistral-dataset-generation

# Install packages

In [ ]:
%%time
from IPython.display import clear_output

!pip install faker

! pip install -q -U transformers
! pip install -q -U accelerate
! pip install -q -U bitsandbytes

! pip install -qq -U langchain

clear_output()

# Load libraries

In [ ]:
%%time

import sys, random, string, re, time, os
import warnings
warnings.filterwarnings("ignore")
import gc
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch

### transformers
import transformers
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

### quantization
import bitsandbytes as bnb

### langchain
from langchain.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain import PromptTemplate, LLMChain
from langchain.llms import HuggingFacePipeline
import langchain

from faker import Faker  #generates fake data 
from spacy.lang.en import English

In [ ]:
import torch
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Device: {DEVICE}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"Pytorch {torch.__version__}")

In [ ]:
import torch, random
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SEED = 42
# Seed the same seed to all 
def seed_everything(seed=42):
    Faker.seed(0)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(SEED)

In [ ]:
import ctypes, gc, torch
libc = ctypes.CDLL("libc.so.6")
def clear_memory():
    libc.malloc_trim(0)
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
print('torch version: ', torch.__version__)
print(f'transformers version: {transformers.__version__}')
print(f'bnb version: {bnb.__version__}')
print(f'langchain version: {langchain.__version__}')

# Configs

In [ ]:
class CFG:
    
    ### model
    MODEL_PATH = '/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1'    
    

# Load LLM `Mistral-7b-instruct-v0.1-hf` to generate PII


In [ ]:
def load_model():
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_flash_sdp(False)
    
    ### quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type = "nf4",
        bnb_4bit_compute_dtype = torch.float16,
        bnb_4bit_use_double_quant = True,
        llm_int8_enable_fp32_cpu_offload = True,
    )
#     tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    
    ### tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        CFG.MODEL_PATH,
        trust_remote_code = True, 
        use_fast=True
    )
    
    ### model
    model = AutoModelForCausalLM.from_pretrained(
        CFG.MODEL_PATH,
        quantization_config = bnb_config,
#         torch_dtype=torch.bfloat16,
        device_map = "auto",
        trust_remote_code = True,
    #     attn_implementation = 'flash_attention_2',
    )
    
    return model, tokenizer

In [ ]:
%%time

model, tokenizer = load_model()

# URL_PERSONAL

In [ ]:
def generate_url():
    
    URL_TYPES = ["social media page", "website"]
    url_type = random.choice(URL_TYPES)
    
    SMEDIA_TYPES = ['LinkedIn','YouTube','Instagram','GitHub','Facebook','Twitter','TikTok']
    smedia_type = random.choice(SMEDIA_TYPES)
    smedia_type = '' if url_type == "website" else smedia_type
    
    if url_type == "website":
    
        prompt_template  = """<s>[INST]

                            Here are some examples from the original text which I am trying to generate more synthetic data for:

                            - https://oconnell-townsend.com/wp-content/categorieshomepage.html
                            - http://www.burns-lopez.com/categories/appabout.asp
                            - https://www.hall.biz/wp-contenthome.html
                            - http://jones-mendoza.com/blog/search/searchprivacy.php
                            - tps://mcdonald-pope.com/categorycategory.jsp

                            Generate a url for a personal {url_type} {smedia_type}

                            - Do not include generic things like "username", "name" etc. in the url

                            - Make sure the url has multiple pages or sections like in the examples

                            - Return only one url suggestion and nothing else 

                            Give me a response as follows

                            Generated_url: <insert-generated-url>


                            [/INST]"""
    
    else:
        
        prompt_template  = """<s>[INST]

                            Here are some examples from the original text which I am trying to generate more synthetic data for:

                            - https://youtu.be/rFD2lJuvace
                            - https://www.linkedin.com/in/mmartinez
                            - tps://www.facebook.com/bclark

                            Generate a url for a personal {url_type} {smedia_type}

                            - Do not include generic things like "username", "name" etc. in the url

                            - Return only one url suggestion and nothing else 

                            Give me a response as follows

                            Generated_url: <insert-generated-url>


                            [/INST]"""

    # Fill in prompt with PII
    prompt = prompt_template.format(url_type=url_type,
                                    smedia_type=smedia_type
                                   )

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    
    # Generate the outputs from prompt
    generate_ids = model.generate(**inputs, 
                                  max_new_tokens=100,
                                  do_sample=True,
                                  temperature=0.9,
                                  top_p=0.95,
                                  top_k=40,
                                  repetition_penalty=1.1,
                                  pad_token_id=tokenizer.eos_token_id
                                 )
    # Decode the generated output
    generated_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True,
                                             clean_up_tokenization_spaces=False)[0]
    generated_text = generated_text.split('[/INST] ')[1]
    
    generated_text = generated_text[generated_text.find("_url:")+6:].strip().strip("<").strip(">")

    return generated_text
    

# ID_NUM

In [ ]:
def generate_idnum():
    
    prompt  = """<s>[INST]

                        Here are some examples from the original text which I am trying to generate more synthetic data for:

                        - 982645662261
                        - 409046248321
                        - nMFtUVxSUI|33529258
                        - 06EYD876
                        - VZ:775Y6A5764
                        
                        Generate a number or sequence of characters that could be used to identify a student, such as a student ID or a social security number.
                        
                        Return only one id number suggestion and nothing else
                        
                        Give me a response as follows
                        
                        Generated_id: <insert-generated-id>

                        [/INST]"""

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    
    # Generate the outputs from prompt
    generate_ids = model.generate(**inputs, 
                                  max_new_tokens=30,
                                  do_sample=True,
                                  temperature=0.9,
                                  top_p=0.95,
                                  top_k=40,
                                  repetition_penalty=1.1,
                                  pad_token_id=tokenizer.eos_token_id
                                 )
    # Decode the generated output
    generated_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True,
                                             clean_up_tokenization_spaces=False)[0]
    generated_text = generated_text.split('[/INST] ')[1]
    
    generated_text = generated_text[generated_text.find("_id:")+5:].strip()

    return generated_text
    

# EMAIL

In [ ]:
def generate_email():
    
    prompt  = """<s>[INST]

                        Here are some examples from the original text which I am trying to generate more synthetic data for:

                        - agood@gmail.com
                        - vmartinez@hotmail.com
                        - catherine19@hotmail.com
                        - nbarker@hotmail.com
                        - hbrown@yahoo.com
                        - johnsondavid@hotmail.com
                        
                        Generate a student’s email address.
                        
                        Return only one email address suggestion and nothing else
                        
                        Give me a response as follows
                        
                        Generated_email: <insert-generated-email>

                        [/INST]"""

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    
    # Generate the outputs from prompt
    generate_ids = model.generate(**inputs, 
                                  max_new_tokens=100,
                                  do_sample=True,
                                  temperature=0.9,
                                  top_p=0.95,
                                  top_k=40,
                                  repetition_penalty=1.1,
                                  pad_token_id=tokenizer.eos_token_id
                                 )
    # Decode the generated output
    generated_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True,
                                             clean_up_tokenization_spaces=False)[0]
    generated_text = generated_text.split('[/INST] ')[1]
    
    generated_text = generated_text[generated_text.find("_email:")+8:].strip()

    return generated_text
    

# PHONE_NUM

In [ ]:
def generate_phnum():
    
    prompt  = """<s>[INST]

                        Here are some examples from the original text which I am trying to generate more synthetic data for:

                        - (320)202-0688x95843
                        - (820)913-3241x894
                        - (223)392-2765
                        - 410.526.1667
                        
                        Generate a phone number associated with a student.
                        
                        Return only one phone number suggestion and nothing else
                        
                        Give me a response as follows
                        
                        Generated_phnum: <insert-generated-phone-number>

                        [/INST]"""

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    
    # Generate the outputs from prompt
    generate_ids = model.generate(**inputs, 
                                  max_new_tokens=100,
                                  do_sample=True,
                                  temperature=0.9,
                                  top_p=0.95,
                                  top_k=40,
                                  repetition_penalty=1.1,
                                  pad_token_id=tokenizer.eos_token_id
                                 )
    # Decode the generated output
    generated_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True,
                                             clean_up_tokenization_spaces=False)[0]
    generated_text = generated_text.split('[/INST] ')[1]
    
    generated_text = generated_text[generated_text.find("_phnum:")+8:].strip()

    return generated_text
    

# STREET_ADDRESS

In [ ]:
def generate_addr():
    
    prompt  = """<s>[INST]

                        Here are some examples from the original text which I am trying to generate more synthetic data for:

                        - 591 Smith Centers Apt. 656 Joshuamouth, RI 95963
                        - 743 Erika Bypass Apt. 419 Andreahaven, IL 54207
                        
                        Generate a full or partial street address that is associated with the student, such as their home address.
                        
                        Return only one street address suggestion and nothing else. Do not make the address too generic (123 Anytown, etc). Use real city, street, etc
                        
                        Give me a response as follows
                        
                        Generated_addr: <insert-generated-street-address>

                        [/INST]"""

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    
    # Generate the outputs from prompt
    generate_ids = model.generate(**inputs, 
                                  max_new_tokens=100,
                                  do_sample=True,
                                  temperature=0.9,
                                  top_p=0.95,
                                  top_k=40,
                                  repetition_penalty=1.1,
                                  pad_token_id=tokenizer.eos_token_id
                                 )
    # Decode the generated output
    generated_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True,
                                             clean_up_tokenization_spaces=False)[0]
    generated_text = generated_text.split('[/INST] ')[1]
    
    generated_text = generated_text[generated_text.find("_addr:")+7:].strip()

    return generated_text
    

# USERNAME

In [ ]:
def generate_username():
    
    prompt  = """<s>[INST]

                        Here are some examples from the original text which I am trying to generate more synthetic data for:

                        - castanedagabriel
                        - fdixon
                        - meyermichelle
                        - jacob59
                        - holmespatrick
                        
                        Generate a student's username on any platform.
                        
                        Return only one username suggestion and nothing else
                        
                        Give me a response as follows
                        
                        Generated_uname: <insert-generated-username>

                        [/INST]"""

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    
    # Generate the outputs from prompt
    generate_ids = model.generate(**inputs, 
                                  max_new_tokens=30,
                                  do_sample=True,
                                  temperature=0.9,
                                  top_p=0.95,
                                  top_k=40,
                                  repetition_penalty=1.1,
                                  pad_token_id=tokenizer.eos_token_id
                                 )
    # Decode the generated output
    generated_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True,
                                             clean_up_tokenization_spaces=False)[0]
    generated_text = generated_text.split('[/INST] ')[1]
    
    generated_text = generated_text[generated_text.find("_uname:")+8:].strip()

    return generated_text
    

# NAME_STUDENT

In [ ]:
def generate_name():
    
    prompt  = """<s>[INST]

                        Here are some examples from the original text which I am trying to generate more synthetic data for:

                        - Basavaraju Aakash Kumar
                        - Sameh
                        - Ronnie Shahed
                        - Jan Peters
                        - Sarah Calixto
                        - Agim Krieger
                        - Yaser Mostafa
                        - Giorgia Piccolo
                        - Miguel Perez
                        
                        Generate a full or partial name of a student that is not necessarily the author of the essay. This excludes instructors, authors, and other person names.
                        
                        Return only one name suggestion and nothing else
                        
                        Give me a response as follows
                        
                        Generated_name: <insert-generated-username>

                        [/INST]"""

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    
    # Generate the outputs from prompt
    generate_ids = model.generate(**inputs, 
                                  max_new_tokens=30,
                                  do_sample=True,
                                  temperature=0.9,
                                  top_p=0.95,
                                  top_k=40,
                                  repetition_penalty=1.1,
                                  pad_token_id=tokenizer.eos_token_id
                                 )
    # Decode the generated output
    generated_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True,
                                             clean_up_tokenization_spaces=False)[0]
    generated_text = generated_text.split('[/INST] ')[1]
    
    generated_text = generated_text[generated_text.find("_name:")+7:].strip()

    return generated_text

# Generate PII dataframe

In [ ]:
pii_count_dict = {
            'NAME_STUDENT': 2000,
            'EMAIL': 80,
            'USERNAME': 20,
            'ID_NUM': 150,
            'PHONE_NUM': 40,
            'URL_PERSONAL': 200,
            'STREET_ADDRESS': 40,
            }

pii_func_dict = {
            'NAME_STUDENT': 'generate_name',
            'EMAIL': 'generate_email',
            'USERNAME': 'generate_username',
            'ID_NUM': 'generate_idnum',
            'PHONE_NUM': 'generate_phnum',
            'URL_PERSONAL': 'generate_url',
            'STREET_ADDRESS': 'generate_addr',
            }

pii_maxlen_dict = {
            'NAME_STUDENT': 30,
            'EMAIL': 50,
            'USERNAME': 30,
            'ID_NUM': 30,
            'PHONE_NUM': 20,
            'URL_PERSONAL': 200,
            'STREET_ADDRESS': 100,
            }

In [ ]:
pii_type = []
pii_id = []

for pii in tqdm(pii_count_dict.keys()):
    
    print(pii, pii_count_dict[pii])
    
    for i in tqdm(range(pii_count_dict[pii])):
        
        while True:
        
            expii = globals()[pii_func_dict[pii]]()
        
            if len(expii) > pii_maxlen_dict[pii]:
                continue
            
            break
        
        pii_type.append(pii)
        pii_id.append(expii)
        

pii_df = pd.DataFrame({'pii_type': pii_type,
                      'pii_id': pii_id})

pii_df.shape

# Save dataframe

In [ ]:
pii_df.head()

In [ ]:
pii_df.to_csv("pii_mistral.csv", index=False)